# Preprocessing - Member 3: Feature Selection
## Online Shopper Purchase Likelihood Prediction System

This notebook uses Member 1's engineered and log-transformed training data. Feature selection is evaluated on the training set with stratified cross-validation; the held-out test set is not inspected here.

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from src.preprocessing.feature_selection import (
    compare_feature_drop,
    correlation_pairs,
    cross_validated_permutation_importance,
)

## Load the training handoff

These files were created after validation, duplicate removal, the stratified split, feature engineering, and log transformation. The test set remains untouched.

In [2]:
X_train = pd.read_csv('../data/processed/X_train_engineered.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze('columns').astype(int)

categorical_columns = [
    'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType',
    'VisitorType', 'Weekend', 'VisitorType_Weekend',
]
numeric_columns = [column for column in X_train.columns if column not in categorical_columns]
X_train_numeric = X_train[numeric_columns].copy()
print('Training rows:', len(X_train_numeric))
print('Numeric features evaluated:', len(numeric_columns))

Training rows: 9764
Numeric features evaluated: 13


## 1. Correlation screening

Correlation identifies potentially redundant numeric features; it does not decide removal. The threshold is a screening rule applied to training data only.

In [3]:
correlated_pairs = correlation_pairs(X_train_numeric, threshold=0.80)
correlated_pairs

,feature_a,feature_b,correlation,absolute_correlation
0,ProductRelated,TotalPages,0.988947,0.988947
1,ProductRelated_Duration,TotalDuration,0.986111,0.986111
2,BounceRates,ExitRates,0.899991,0.899991
3,TotalPages,TotalDuration,0.853646,0.853646
4,ProductRelated,ProductRelated_Duration,0.852326,0.852326
5,ProductRelated_Duration,TotalPages,0.851255,0.851255
6,ProductRelated,TotalDuration,0.836330,0.836330
7,Informational,Informational_Duration,0.823713,0.823713


## 2. Cross-validated feature importance

Permutation importance is measured on validation folds after each model is fitted on its corresponding training fold. This avoids using the test set and gives a model-based check beyond pairwise correlation.

In [4]:
selection_model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

importance = cross_validated_permutation_importance(
    selection_model, X_train_numeric, y_train, scoring='f1', n_repeats=10
)
importance

,feature,importance_mean,importance_std
0,PageValues,0.472295,0.013443
1,ExitRates,0.035904,0.009059
2,BounceRates,0.009004,0.007769
3,SpecialDay,0.005295,0.004552
4,AvgTimePerPage,0.004789,0.002367
5,ProductRelated_Duration,0.004647,0.009732
6,TotalDuration,0.004565,0.007089
7,TotalPages,-0.000654,0.007388
8,ProductRelated,-0.002083,0.005661
9,Informational_Duration,-0.004225,0.003625


## 3. Validate the `ExitRates` drop candidate

`BounceRates` and `ExitRates` were flagged in EDA as redundant candidates. We compare the same estimator and folds with both features retained versus `ExitRates` removed. The higher validated F1 option is retained; correlation alone is not enough to justify deletion.

In [5]:
exit_rates_comparison = compare_feature_drop(
    selection_model, X_train_numeric, y_train, 'ExitRates', scoring='f1'
)
exit_rates_comparison

,feature_set,mean_score,std_score
0,keep,0.659773,0.011748
1,drop,0.655257,0.012153


In [6]:
keep_score = exit_rates_comparison.loc[exit_rates_comparison['feature_set'] == 'keep', 'mean_score'].iloc[0]
drop_score = exit_rates_comparison.loc[exit_rates_comparison['feature_set'] == 'drop', 'mean_score'].iloc[0]
selected_numeric_columns = (
    X_train_numeric.columns.drop('ExitRates').tolist()
    if drop_score > keep_score
    else X_train_numeric.columns.tolist()
)
print('Selected option:', 'drop ExitRates' if drop_score > keep_score else 'keep all numeric features')
print('Selected numeric feature count:', len(selected_numeric_columns))

Selected option: keep all numeric features
Selected numeric feature count: 13


**Justification (Member 3):** Correlation and permutation importance are complementary checks: correlation detects redundancy between numeric inputs, while permutation importance measures the contribution of each feature to held-out validation performance. `ExitRates` is removed only if the cross-validated F1 score improves when it is dropped; otherwise both `BounceRates` and `ExitRates` are retained. The chosen feature list must be passed to the next pipeline stage for encoding/scaling and then applied identically to the test data.